# Notebook 02: Feature Engineering

**Project:** GNN-Based Battery Voltage Predictor  

---

## Overview

This notebook constructs two parallel feature representations for the battery dataset:

### A. Crystal Graph Representation (for CGCNN and M3GNet)

Each crystal structure is converted to a graph where:
- **Nodes** represent atomic sites with a 9-dimensional feature vector:
  - Atomic number (normalized), Pauling electronegativity, ionic radius,
    group number, period number, is-metal flag, is-transition-metal flag,
    is-alkali flag, is-alkaline-earth flag
- **Edges** represent bonds within a 5.0 A cutoff (CrystalNN neighbor finding);
  each edge carries a 64-dimensional Gaussian basis function encoding of the
  interatomic distance (centers from 0.5 to 5.0 A)

Processed graphs are saved as PyG DataLoader-compatible pickle files.

### B. Compositional Descriptors (for Random Forest baseline)

Matminer ElementProperty (MAGPIE preset) and IonProperty featurizers
produce a fixed-length descriptor vector from the chemical formula.
No structural information is required, making this fast to compute
and interpretable via feature importance.

All processed features are saved to `data/` for use in Notebook 03.

In [ ]:
import sys
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm import tqdm

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import VoltageGraphDataset, build_matminer_features
from src.utils import get_atom_features, gaussian_basis, structure_to_graph
from src.evaluate import PALETTE

DATA_DIR = project_root / 'data'
RESULTS_DIR = project_root / 'results'

# Load splits from Notebook 01
with open(DATA_DIR / 'splits.pkl', 'rb') as f:
    splits = pickle.load(f)

train_entries = splits['train']
val_entries   = splits['val']
test_entries  = splits['test']

print(f'Loaded: {len(train_entries)} train / {len(val_entries)} val / {len(test_entries)} test')

## 1. Atom Feature Inspection

Before building graphs, we verify the atom feature vectors for a few representative
species encountered in Li battery cathodes: Li, Mn, Fe, Co, Ni, P, O, S, F.

The 9-dimensional feature vector is printed alongside the underlying property values.

In [ ]:
from pymatgen.core.periodic_table import Element

test_elements = ['Li', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'P', 'O', 'S', 'F', 'Si']
feat_labels = [
    'Z (norm)', 'Electro. (norm)', 'Ionic r (norm)',
    'Group (norm)', 'Period (norm)', 'is_metal',
    'is_TM', 'is_alkali', 'is_alkaline'
]

rows = []
for sym in test_elements:
    feats = get_atom_features(sym)
    rows.append([sym] + feats)

df_feats = pd.DataFrame(rows, columns=['Element'] + feat_labels)
df_feats = df_feats.set_index('Element')
print(df_feats.round(3).to_string())

In [ ]:
# Visualize Gaussian basis encoding for several distances
distances = [1.5, 2.0, 2.5, 3.0, 4.0]
centers = np.linspace(0.5, 5.0, 64)

fig, ax = plt.subplots(figsize=(8, 3.5))
for d in distances:
    gbf = gaussian_basis(d, n_bins=64, d_min=0.5, d_max=5.0)
    ax.plot(centers, gbf, label=f'd = {d} A')

ax.set_xlabel('Gaussian Basis Center (A)')
ax.set_ylabel('Activation')
ax.set_title('Gaussian Basis Function Encoding of Bond Distance')
ax.legend(frameon=False, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig02_gaussian_basis.png')
plt.show()

## 2. Crystal Graph Construction

We build PyG graph objects for all entries in each split using `VoltageGraphDataset`.
This wraps the `structure_to_graph()` utility and stores graphs in an InMemoryDataset.

Expected timing: approximately 2 to 5 seconds per 1,000 structures (CPU-bound).

If graphs are already cached, we load from disk to save time.

In [ ]:
GRAPH_CACHE = {
    'train': DATA_DIR / 'train_graphs.pkl',
    'val':   DATA_DIR / 'val_graphs.pkl',
    'test':  DATA_DIR / 'test_graphs.pkl',
}

datasets = {}

for split_name, entries_list in [('train', train_entries), ('val', val_entries), ('test', test_entries)]:
    cache_path = GRAPH_CACHE[split_name]
    if cache_path.exists():
        print(f'Loading cached {split_name} graphs from {cache_path}')
        datasets[split_name] = VoltageGraphDataset.from_processed(str(cache_path))
    else:
        print(f'Building {split_name} graphs ({len(entries_list)} entries)...')
        t0 = time.time()
        ds = VoltageGraphDataset(entries_list, use_charged=True, cutoff=5.0, n_gbf_bins=64)
        elapsed = time.time() - t0
        print(f'  Built {len(ds)} graphs in {elapsed:.1f}s')
        ds.save_processed(str(cache_path))
        datasets[split_name] = ds

train_ds = datasets['train']
val_ds   = datasets['val']
test_ds  = datasets['test']

print(f'\nDataset sizes: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')

In [ ]:
# Inspect a single graph
sample = train_ds[0]
print('Sample graph properties:')
print(f'  Number of nodes:      {sample.num_nodes}')
print(f'  Node feature shape:   {sample.x.shape}     (N x 9)')
print(f'  Number of edges:      {sample.edge_index.shape[1]}')
print(f'  Edge feature shape:   {sample.edge_attr.shape}  (E x 64)')
print(f'  Target voltage:       {sample.y.item():.4f} V')
print(f'  Formula:              {sample.formula}')
print(f'  Chemistry family:     {sample.chemistry_family}')
print(f'\nNode features (first 3 atoms):')
print(sample.x[:3].numpy().round(4))

In [ ]:
# Graph size statistics
n_nodes = [train_ds[i].num_nodes for i in range(len(train_ds))]
n_edges = [train_ds[i].edge_index.shape[1] for i in range(len(train_ds))]
voltages = [train_ds[i].y.item() for i in range(len(train_ds))]

print(f'Nodes per graph: min={min(n_nodes)}, max={max(n_nodes)}, mean={np.mean(n_nodes):.1f}')
print(f'Edges per graph: min={min(n_edges)}, max={max(n_edges)}, mean={np.mean(n_edges):.1f}')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

axes[0].hist(n_nodes, bins=40, color='#2196F3', alpha=0.8, edgecolor='none')
axes[0].set_xlabel('Atoms per Unit Cell')
axes[0].set_ylabel('Count')
axes[0].set_title('Graph Size Distribution')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].hist(n_edges, bins=40, color='#4CAF50', alpha=0.8, edgecolor='none')
axes[1].set_xlabel('Bonds per Unit Cell (cutoff 5.0 A)')
axes[1].set_ylabel('Count')
axes[1].set_title('Edge Count Distribution')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig02_graph_size_distribution.png')
plt.show()

## 3. Matminer Compositional Descriptors (Random Forest Baseline)

The MAGPIE (Materials Agnostic Platform for Informatics and Exploration) descriptor set
from `matminer.featurizers.composition.ElementProperty` computes statistics
(mean, std, min, max, range) over element properties for each composition.

Additionally `IonProperty` features capture charge balance and ionic character.

The resulting feature matrix has approximately 132 columns. These descriptors
are composition-only (no structural information), which limits accuracy
but enables fast inference and interpretable feature importance analysis.

In [ ]:
MATMINER_CACHE = DATA_DIR / 'matminer_features.pkl'

if MATMINER_CACHE.exists():
    print(f'Loading cached matminer features from {MATMINER_CACHE}')
    with open(MATMINER_CACHE, 'rb') as f:
        mm_data = pickle.load(f)
    X_train_mm, y_train_mm, feature_names = mm_data['train']
    X_val_mm,   y_val_mm,   _             = mm_data['val']
    X_test_mm,  y_test_mm,  _             = mm_data['test']
else:
    print('Computing matminer features (this may take 5 to 10 minutes)...')
    X_train_mm, y_train_mm, feature_names = build_matminer_features(train_entries)
    X_val_mm,   y_val_mm,   _             = build_matminer_features(val_entries)
    X_test_mm,  y_test_mm,  _             = build_matminer_features(test_entries)

    mm_data = {
        'train': (X_train_mm, y_train_mm, feature_names),
        'val':   (X_val_mm,   y_val_mm,   feature_names),
        'test':  (X_test_mm,  y_test_mm,  feature_names),
    }
    with open(MATMINER_CACHE, 'wb') as f:
        pickle.dump(mm_data, f)

print(f'Train feature matrix: {X_train_mm.shape}')
print(f'Number of descriptors: {len(feature_names)}')
print(f'\nFirst 10 feature names:')
for n in feature_names[:10]:
    print(f'  {n}')

In [ ]:
# Check feature variance; drop near-zero variance features
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=1e-6)
X_train_filtered = vt.fit_transform(X_train_mm)
X_val_filtered   = vt.transform(X_val_mm)
X_test_filtered  = vt.transform(X_test_mm)
feature_names_filtered = [feature_names[i] for i in vt.get_support(indices=True)]

print(f'Features before variance filter: {X_train_mm.shape[1]}')
print(f'Features after variance filter:  {X_train_filtered.shape[1]}')

# Save filtered arrays
with open(DATA_DIR / 'matminer_filtered.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train_filtered, 'y_train': y_train_mm,
        'X_val':   X_val_filtered,   'y_val':   y_val_mm,
        'X_test':  X_test_filtered,  'y_test':  y_test_mm,
        'feature_names': feature_names_filtered,
    }, f)
print(f'Saved filtered matminer features.')

In [ ]:
# Compute correlation of each feature with voltage
corrs = np.array([abs(np.corrcoef(X_train_filtered[:, i], y_train_mm)[0, 1])
                  for i in range(X_train_filtered.shape[1])])
corrs = np.nan_to_num(corrs)
top_idx = np.argsort(corrs)[::-1][:15]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(range(15),
               corrs[top_idx],
               color='#4CAF50', alpha=0.85, edgecolor='none')
ax.set_yticks(range(15))
ax.set_yticklabels([feature_names_filtered[i] for i in top_idx], fontsize=8.5)
ax.set_xlabel('Absolute Pearson Correlation with Voltage')
ax.set_title('Top 15 Matminer Features by Correlation with Target')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig02_feature_correlation.png')
plt.show()
print('Saved: fig02_feature_correlation.png')

In [ ]:
print('Feature engineering complete.')
print(f'  Crystal graphs (PyG): {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')
print(f'  Matminer features:    {X_train_filtered.shape[1]} descriptors')
print(f'\nProceed to 03_model_training.ipynb')